====================== Generative Adversarial Networks ======================

#### GAN Core Idea

A Generative Adversarial Network (GAN) sets up a **two-player game** between:
- **Generator `G(z)`**: transforms random noise `z ~ p(z)` into synthetic samples `G(z)`.
- **Discriminator `D(x)`**: outputs how likely `x` is from real data rather than generated.

!["GAN"](./Images/20/GAN.png)

Intuition:
- `D` is a learned critic that gives gradients about "how fake" samples are.
- `G` uses those gradients to move generated samples toward the real data distribution.

Adversarial objective:
$$
\min_G \max_D\; \mathbb{E}_{x\sim p_{\text{data}}}[\log D(x)] + \mathbb{E}_{z\sim p(z)}[\log(1 - D(G(z)))]
$$

Equivalent practical losses in this notebook:
- Discriminator loss: classify real as `1`, fake as `0`.
- Generator loss: push discriminator outputs on fake samples toward `1`.

Why this works:
- If `D` is weak, `G` receives poor supervision.
- If `D` is too strong, `G` gradients can vanish.
- So GAN training is about keeping both players in a useful learning regime.

In this notebook you first see a **toy 2D Gaussian-style example** (easy to visualize), then a full **DCGAN image model**.

In [ ]:
# Core dependencies for GAN experiments in this chapter.
# `d2l` provides helper utilities for plotting, timing, and training loops.
%matplotlib inline
import torch
from torch import nn
from d2l import torch as d2l

In [ ]:
# Generate Some “Real” Data
X = torch.normal(0.0, 1, (1000, 2))
A = torch.tensor([[1, 2], [-0.1, 0.5]])
b = torch.tensor([1, 2])
data = torch.matmul(X, A) + b

In [ ]:
# This synthetic "real" dataset is: data = X @ A + b, with X ~ N(0, I).
# - b sets the mean (shifts the cloud center).
# - A sets the geometry (rotation/stretch), so Cov(data) = A^T A
# Why this matters for GAN: in this 2D Gaussian toy setup, matching data means
# matching both center (mean) and shape/orientation/spread (covariance).
#
# Derivation (row-vector form):
# Let Y = X @ A + b, where rows of X are samples.
# 1) Cov(X @ A + b) = Cov(X @ A)      (adding constant b does not change covariance)
# 2) Cov(X @ A) = A^T Cov(X) A       (linear transform rule for row vectors)
# 3) Cov(X) = I                       (since X ~ N(0, I))
# Therefore, Cov(Y) = A^T I A = A^T A.
#
# Reading entries of Cov(Y) = [[c11, c12], [c21, c22]]:
# - c11: Var(Y1), c22: Var(Y2)
# - c12 (= c21): Cov(Y1, Y2), i.e., relationship between the two output dimensions.
xy = data[:100].detach().numpy()
c = list(range(len(xy)))
d2l.plt.figure(figsize=(5, 4))
d2l.plt.scatter(xy[:, 0], xy[:, 1], c=c, cmap='viridis', s=28)
d2l.plt.colorbar(label='sample index')
d2l.plt.xlabel('column 0 (Y1)')
d2l.plt.ylabel('column 1 (Y2)')
d2l.plt.title('2D samples: x=col0, y=col1')
cov = torch.matmul(A.T, A)
print(f'The covariance matrix is\n{cov}')
print(f'Var(Y1)={cov[0,0]:.2f}, Var(Y2)={cov[1,1]:.2f}, Cov(Y1,Y2)={cov[0,1]:.2f}')

##### Intuition for `Var(Y1)`, `Var(Y2)`, `Cov(Y1, Y2)`

For each sample row, let `X = [x1, x2]` with independent `x1, x2 ~ N(0, 1)`, and
`Y = X @ A + b` with

`A = [[1, 2], [-0.1, 0.5]]`.

So:
- `Y1 = 1*x1 + (-0.1)*x2 + b1`
- `Y2 = 2*x1 + 0.5*x2 + b2`

`b` only shifts the mean and does not change variance.
For independent unit-variance inputs:
`Var(a*x1 + c*x2) = a^2 + c^2`.

Therefore:
- `Var(Y1) = 1^2 + (-0.1)^2 = 1.01` (close to 1)
- `Var(Y2) = 2^2 + 0.5^2 = 4.25`

**standard deviation**: `Std(Y2) = sqrt(4.25) ≈ 2.06`.

So the right intuition is: `Y2` has about twice the spread in standard deviation, but about four times the variance.

Intuition for the cross term (`Cov(Y1, Y2)`):
- For independent `x1, x2`,
  `Cov(a*x1 + c*x2, b*x1 + d*x2) = a*b + c*d`.
- Here, `Cov(Y1, Y2) = 1*2 + (-0.1)*0.5 = 1.95`.
- It is positive because both `Y1` and `Y2` strongly share the `x1` direction with positive weights (`1` and `2`), and that dominates the small negative contribution from `x2` (`-0.05`).

In [ ]:
# Pack the 2D toy samples into mini-batches for alternating D/G updates.
batch_size = 8
data_iter = d2l.load_array((data,), batch_size)

In [ ]:
# Generator
# Our generator network will be the simplest network possible - a single layer linear model. 
# This is since we will be driving that linear network with a Gaussian data generator. 
# Hence, it literally only needs to learn the parameters to fake things perfectly
net_G = nn.Sequential(nn.Linear(2, 2))

In [ ]:
# Discriminator
# For the discriminator we will be a bit more discriminating:
# we will use an MLP with 3 layers to make things a bit more interesting.
net_D = nn.Sequential(
    nn.Linear(2, 5), nn.Tanh(),
    nn.Linear(5, 3), nn.Tanh(),
    nn.Linear(3, 1))

In [ ]:
# Training

#@save
def update_D(X, Z, net_D, net_G, loss, trainer_D):
    """Update discriminator."""
    batch_size = X.shape[0]
    ones = torch.ones((batch_size,), device=X.device)
    zeros = torch.zeros((batch_size,), device=X.device)
    trainer_D.zero_grad()
    real_Y = net_D(X)
    # Z is a fake initial data, pass through the generator to try to produce a fake X
    fake_X = net_G(Z)
    # Do not need to compute gradient for `net_G`, detach it from computing gradients.
    fake_Y = net_D(fake_X.detach())
    loss_D = (loss(real_Y, ones.reshape(real_Y.shape)) + loss(fake_Y, zeros.reshape(fake_Y.shape))) / 2
    loss_D.backward()
    trainer_D.step()
    return loss_D

In [ ]:
#@save
def update_G(Z, net_D, net_G, loss, trainer_G):
    """Update generator."""
    batch_size = Z.shape[0]
    ones = torch.ones((batch_size,), device=Z.device)
    trainer_G.zero_grad()
    # We could reuse `fake_X` from `update_D` to save computation
    fake_X = net_G(Z)
    # Recomputing `fake_Y` is needed since `net_D` is changed
    fake_Y = net_D(fake_X)
    loss_G = loss(fake_Y, ones.reshape(fake_Y.shape))
    loss_G.backward()
    trainer_G.step()
    return loss_G

##### How `update_D` and `update_G` map to the GAN game

This notebook's two update functions implement one adversarial training step:
1. `update_D(...)`
- Uses real batch `X` and fake batch `G(Z)`.
- Calls `fake_X.detach()` so discriminator update does not backprop into generator.
- Optimizes discriminator to separate real (`1`) from fake (`0`).

2. `update_G(...)`
- Recomputes `fake_X = G(Z)` and evaluates `D(fake_X)` with updated discriminator.
- Optimizes generator so fake samples are classified as real (`1`).

This alternating optimization approximates the minimax objective from D2L section 20.1.
For stability, the model quality should be judged by sample trajectories and diversity, not a single loss curve.

In [ ]:
# Alternate discriminator and generator updates each mini-batch, then
# visualize how generated points move toward the real 2D distribution.
def train(net_D, net_G, data_iter, num_epochs, lr_D, lr_G, latent_dim, data):
    loss = nn.BCEWithLogitsLoss(reduction='sum')
    for w in net_D.parameters():
        nn.init.normal_(w, 0, 0.02)
    for w in net_G.parameters():
        nn.init.normal_(w, 0, 0.02)
    trainer_D = torch.optim.Adam(net_D.parameters(), lr=lr_D)
    trainer_G = torch.optim.Adam(net_G.parameters(), lr=lr_G)
    animator = d2l.Animator(xlabel='epoch', ylabel='loss',
                            xlim=[1, num_epochs], nrows=2, figsize=(5, 5),
                            legend=['discriminator', 'generator'])
    animator.fig.subplots_adjust(hspace=0.3)
    for epoch in range(num_epochs):
        # Train one epoch
        timer = d2l.Timer()
        metric = d2l.Accumulator(3)  # loss_D, loss_G, num_examples
        for (X,) in data_iter:
            batch_size = X.shape[0]
            Z = torch.normal(0, 1, size=(batch_size, latent_dim))
            metric.add(update_D(X, Z, net_D, net_G, loss, trainer_D),
                       update_G(Z, net_D, net_G, loss, trainer_G),
                       batch_size)
        # Visualize generated examples
        Z = torch.normal(0, 1, size=(100, latent_dim))
        fake_X = net_G(Z).detach().numpy()
        animator.axes[1].cla()
        animator.axes[1].scatter(data[:, 0], data[:, 1])
        animator.axes[1].scatter(fake_X[:, 0], fake_X[:, 1])
        animator.axes[1].legend(['real', 'generated'])
        # Show the losses
        loss_D, loss_G = metric[0]/metric[2], metric[1]/metric[2]
        animator.add(epoch + 1, (loss_D, loss_G))
    print(f'loss_D {loss_D:.3f}, loss_G {loss_G:.3f}, '
          f'{metric[2] / timer.stop():.1f} examples/sec')

In [ ]:
# Use a slightly stronger discriminator learning rate on this toy task.
lr_D, lr_G, latent_dim, num_epochs = 0.05, 0.005, 2, 20
train(net_D, net_G, data_iter, num_epochs, lr_D, lr_G,
      latent_dim, data[:100].detach().numpy())

======= Deep Convolutional Generative Adversarial Networks =======

#### DCGAN Practical Recipe

This section adapts GANs to images by using convolutional inductive bias.

Generator side (`net_G`):
- Start from latent tensor `(batch, latent_dim, 1, 1)`.
- Repeated `ConvTranspose2d + BatchNorm + ReLU` blocks upsample spatial resolution.
- Final `Tanh` outputs image pixels in `[-1, 1]` (matching normalized training images).

Discriminator side (`net_D`):
- Input image `(batch, channels, 64, 64)`.
- Repeated `Conv2d + BatchNorm + LeakyReLU` blocks downsample and increase channels.
- Final conv outputs a single logit per image (real/fake evidence).

Training details used here:
- `BCEWithLogitsLoss` for numerical stability (logit-level binary classification).
- Adam with `betas=(0.5, 0.999)` to improve adversarial stability.
- Weight initialization with small normal noise (`std=0.02`).

How to interpret losses:
- Lower `loss_D` is not always better; an overly strong `D` can hurt `G`.
- Lower `loss_G` is not always better either; sample quality/diversity matters more.
- Always inspect generated images over epochs, not just scalar losses.

Common failure modes:
- **Mode collapse**: many latent inputs map to similar outputs.
- **Oscillation**: `D` and `G` losses cycle without visual improvement.
- **Gradient starvation**: one network dominates and the other stops improving.

In [ ]:
# Imports for DCGAN on image data.
# torchvision is used for dataset loading and image transforms.
import warnings
import torch
import torchvision
from torch import nn
from d2l import torch as d2l

In [ ]:
# Register and load the Pokemon dataset used in the D2L DCGAN example.
#@save
d2l.DATA_HUB['pokemon'] = (d2l.DATA_URL + 'pokemon.zip',
                           'c065c0e2593b8b161a2d7873e42418bf6a21106c')

data_dir = d2l.download_extract('pokemon')
pokemon = torchvision.datasets.ImageFolder(data_dir)

In [ ]:
# We resize each image into 64*64. The ToTensor transformation will project the pixel value into [0,1], 
# while our generator will use the tanh function to obtain outputs in [-1, 1]. 
# Therefore we normalize the data with  mean and  standard deviation to match the value range.

batch_size = 256
transformer = torchvision.transforms.Compose([
    torchvision.transforms.Resize((64, 64)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(0.5, 0.5)
])
pokemon.transform = transformer
data_iter = torch.utils.data.DataLoader(
    pokemon, batch_size=batch_size,
    shuffle=True, num_workers=d2l.get_dataloader_workers())

In [ ]:
# Let’s visualize the first 20 images.
warnings.filterwarnings('ignore')
d2l.set_figsize((4, 4))
for X, y in data_iter:
    imgs = X[:20,:,:,:].permute(0, 2, 3, 1)/2+0.5
    d2l.show_images(imgs, num_rows=4, num_cols=5)
    break

In [ ]:
# Generator block: transposed convolution unsamples(enlarges) spatial size, then
# BatchNorm + ReLU stabilizes and accelerates generator training.
class G_block(nn.Module):
    def __init__(self, out_channels, in_channels=3, kernel_size=4, strides=2,
                 padding=1, **kwargs):
        super(G_block, self).__init__(**kwargs)
        self.conv2d_trans = nn.ConvTranspose2d(in_channels, out_channels,
                                kernel_size, strides, padding, bias=False)
        self.batch_norm = nn.BatchNorm2d(out_channels)
        self.activation = nn.ReLU()

    def forward(self, X):
        return self.activation(self.batch_norm(self.conv2d_trans(X)))

In [ ]:
# Shape check: with stride=2 and padding=1, spatial size doubles (16 -> 32).
x = torch.zeros((2, 3, 16, 16))
g_blk = G_block(20)
g_blk(x).shape

In [ ]:
# Shape check: first generator block maps 1x1 latent seed to 4x4 feature maps.
x = torch.zeros((2, 3, 1, 1))
g_blk = G_block(20, strides=1, padding=0)
g_blk(x).shape

In [ ]:
# DCGAN generator: progressively upsample noise (1x1) into a 64x64 RGB image.
n_G = 64
net_G = nn.Sequential(
    G_block(in_channels=100, out_channels=n_G*8, strides=1, padding=0), # Output: (64 * 8, 4, 4)               # Output: (64 * 8, 4, 4)
    G_block(in_channels=n_G*8, out_channels=n_G*4), # Output: (64 * 4, 8, 8)
    G_block(in_channels=n_G*4, out_channels=n_G*2), # Output: (64 * 2, 16, 16)
    G_block(in_channels=n_G*2, out_channels=n_G),   # Output: (64, 32, 32)
    nn.ConvTranspose2d(
        in_channels=n_G,
        out_channels=3,
        kernel_size=4,
        stride=2,
        padding=1,
        bias=False), # Output: (3, 64, 64)
    nn.Tanh())  # Output: (3, 64, 64)

##### Q: Why not generate directly from random `(3, 64, 64)`? Why start from `(latent_dim, 1, 1)`?

In DCGAN, the input to the generator is a **latent code** (compact representation), not a noisy image.

- Typical input shape is `(batch, latent_dim, 1, 1)` (for example, `latent_dim=100`).
- The generator then upsamples step by step: `1x1 -> 4x4 -> 8x8 -> ... -> 64x64`.

Why this design is preferred:
- It encourages **coarse-to-fine synthesis**: early layers learn global structure, later layers add local details.
- It is usually **more stable** than injecting high-resolution random noise at pixel level from the start.
- It is **more parameter-efficient**, because the model learns from a low-dimensional latent space.

If we start from random `(3, 64, 64)`:
- Noise is already spread across all pixels, so learning global coherence is harder.
- Training is often noisier and less stable in adversarial settings.

So `(latent_dim, 1, 1)` is an architectural bias that helps GANs learn realistic images more reliably.

##### Q: If the input is always random, how does the generator learn to produce meaningful images?

Great question. The latent vector stays random, but the **generator parameters** are updated every step.

Training loop intuition:
1. Sample random latent vectors `z ~ N(0, I)`.
2. Generator produces fake images `x_fake = G(z)`.
3. Discriminator scores those images (how real/fake they look).
4. Compute generator loss (push `D(G(z))` toward "real").
5. Backpropagate gradient through `D` into `G`, then update `G`'s weights.

So randomness provides diverse inputs, while gradients provide direction.

Another way to view it:
- Early in training, `G` maps most `z` values to poor outputs.
- After many updates, `G` reshapes that mapping so nearby latent points generate structured, coherent images.
- The latent distribution does not become smarter; `G` becomes better at decoding it.

In short: random `z` is the source of variation, and adversarial gradients teach `G` how to turn that variation into meaningful image structure.

In [ ]:
# End-to-end generator shape check: latent tensor -> 64x64 RGB image.
x = torch.zeros((1, 100, 1, 1))
net_G(x).shape

In [ ]:
# Visualize LeakyReLU for different negative slopes used by the discriminator.
alphas = [0, .2, .4, .6, .8, 1]
x = torch.arange(-2, 1, 0.1)
Y = [nn.LeakyReLU(alpha)(x).detach().numpy() for alpha in alphas]
d2l.plot(x.detach().numpy(), Y, 'x', 'y', alphas)

In [ ]:
# Discriminator block: convolution downsamples(decreases) while increasing channels.
# LeakyReLU keeps non-zero gradients for negative activations.
class D_block(nn.Module):
    def __init__(self, out_channels, in_channels=3, kernel_size=4, strides=2,
                padding=1, alpha=0.2, **kwargs):
        super(D_block, self).__init__(**kwargs)
        self.conv2d = nn.Conv2d(in_channels, out_channels, kernel_size,
                                strides, padding, bias=False)
        self.batch_norm = nn.BatchNorm2d(out_channels)
        self.activation = nn.LeakyReLU(alpha, inplace=True)

    def forward(self, X):
        return self.activation(self.batch_norm(self.conv2d(X)))

In [ ]:
# Shape check: one discriminator block downsamples 16x16 to 8x8.
x = torch.zeros((2, 3, 16, 16))
d_blk = D_block(20)
d_blk(x).shape

In [ ]:
# DCGAN discriminator: progressively downsample 64x64 images to one real/fake logit.
n_D = 64
net_D = nn.Sequential(
    D_block(n_D),                                    # Output: (64, 32, 32)
    D_block(in_channels=n_D, out_channels=n_D*2),    # Output: (64 * 2, 16, 16)
    D_block(in_channels=n_D*2, out_channels=n_D*4),  # Output: (64 * 4, 8, 8)
    D_block(in_channels=n_D*4, out_channels=n_D*8),  # Output: (64 * 8, 4, 4)
    nn.Conv2d(in_channels=n_D*8, out_channels=1,
              kernel_size=4, bias=False))            # Output: (1, 1, 1)

##### DCGAN architecture

Shape flow (generator):
- Noise: `(batch, 100, 1, 1)`
- Upsampling blocks: `4x4 -> 8x8 -> 16x16 -> 32x32 -> 64x64`
- Final output: `(batch, 3, 64, 64)`

Shape flow (discriminator):
- Image: `(batch, 3, 64, 64)`
- Downsampling blocks: `64x64 -> 32x32 -> 16x16 -> 8x8 -> 4x4`
- Final logit map: `(batch, 1, 1, 1)`

Why this symmetry helps:
- Generator progressively builds global-to-local image structure.

The generator starts from a tiny latent code (shape like 1x1 with many channels), so early layers can only decide coarse global properties: overall pose, big color regions, rough object layout.
As resolution grows (4x4 -> 8x8 -> ... -> 64x64), later layers add local details: edges, textures, small parts.

- Discriminator progressively extracts local-to-global evidence.

The discriminator does the reverse: it first sees raw pixels and early conv layers detect local patterns (edges, small textures, tiny artifacts).
Deeper layers combine those local cues over larger receptive fields to form global judgments: object consistency, shape plausibility, and whether the whole image looks real.

- The pair forms a natural adversarial feedback loop for image synthesis.

##### Q: If we balance losses of `D` and `G`, does `G` still generate some bad images that `D` can detect?

Yes, this is normal.

Even when training is balanced, GAN optimization is a game, not a guarantee that every sample is perfect.

What usually happens:
1. `G` learns to fool `D` on many latent samples.
2. Some latent inputs still map to lower-quality images.
3. `D` can still classify those weaker samples as fake.

So in practice you often see a mix of strong and weak generations, especially before convergence.

A healthy outcome is:
- `D` is not trivially winning or losing.
- `G` produces many realistic and diverse images.
- But not necessarily perfect quality for every random vector.

In [ ]:
# End-to-end discriminator shape check: RGB image -> single logit map.
x = torch.zeros((1, 3, 64, 64))
net_D(x).shape

In [ ]:
# Full DCGAN training loop: alternate D/G updates on image batches and
# visualize generated samples each epoch to monitor quality/diversity.
def train(net_D, net_G, data_iter, num_epochs, lr, latent_dim,
          device=d2l.try_gpu()):
    loss = nn.BCEWithLogitsLoss(reduction='sum')
    for w in net_D.parameters():
        nn.init.normal_(w, 0, 0.02)
    for w in net_G.parameters():
        nn.init.normal_(w, 0, 0.02)
    net_D, net_G = net_D.to(device), net_G.to(device)
    trainer_hp = {'lr': lr, 'betas': [0.5,0.999]}
    trainer_D = torch.optim.Adam(net_D.parameters(), **trainer_hp)
    trainer_G = torch.optim.Adam(net_G.parameters(), **trainer_hp)
    animator = d2l.Animator(xlabel='epoch', ylabel='loss',
                            xlim=[1, num_epochs], nrows=2, figsize=(5, 5),
                            legend=['discriminator', 'generator'])
    animator.fig.subplots_adjust(hspace=0.3)
    for epoch in range(1, num_epochs + 1):
        # Train one epoch
        timer = d2l.Timer()
        metric = d2l.Accumulator(3)  # loss_D, loss_G, num_examples
        for X, _ in data_iter:
            batch_size = X.shape[0]
            Z = torch.normal(0, 1, size=(batch_size, latent_dim, 1, 1))
            X, Z = X.to(device), Z.to(device)
            metric.add(d2l.update_D(X, Z, net_D, net_G, loss, trainer_D),
                       d2l.update_G(Z, net_D, net_G, loss, trainer_G),
                       batch_size)
        # Show generated examples
        Z = torch.normal(0, 1, size=(21, latent_dim, 1, 1), device=device)
        # Rescale tanh outputs from [-1, 1] back to [0, 1] for visualization.
        fake_x = net_G(Z).permute(0, 2, 3, 1) / 2 + 0.5
        imgs = torch.cat(
            [torch.cat([
                fake_x[i * 7 + j].cpu().detach() for j in range(7)], dim=1)
             for i in range(len(fake_x)//7)], dim=0)
        animator.axes[1].cla()
        animator.axes[1].imshow(imgs)
        # Show the losses
        loss_D, loss_G = metric[0] / metric[2], metric[1] / metric[2]
        animator.add(epoch, (loss_D, loss_G))
    print(f'loss_D {loss_D:.3f}, loss_G {loss_G:.3f}, '
          f'{metric[2] / timer.stop():.1f} examples/sec on {str(device)}')

In [ ]:
latent_dim, lr, num_epochs = 100, 0.005, 20
train(net_D, net_G, data_iter, num_epochs, lr, latent_dim)